In [2]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta
import itertools
import warnings
from sklearn.model_selection import TimeSeriesSplit
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(invalid='ignore', divide='ignore')
# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath(".."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(invalid='ignore', divide='ignore')

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

ticker = 'QQQ'
returns = [5, 10, 15, 20, 25, 35, 45]
lb = 8
cat_cols_all = tags[(tags['Type'] == 'Raw')]['Indicator'].tolist()
windows=[10, 25]

def extract(ticker, returns, lb, cat_cols_all, windows):
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

df = extract(ticker, returns, lb, cat_cols_all, windows)

# Start with must include columns (slope)

In [10]:
def print_metrics(metrics):
        for thresh, metric_values in metrics.items():
            print(f"  Threshold {thresh}: {metric_values}")

def optimize_tests(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_test, y_train, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                metrics[t] = {
                    'PosF1': round(f1_score(y_test_valid, y_pred_valid), 3),
                    'NegF1': round((2*round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3)*round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3))/(round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3) + round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3)),3),
                    "PosTot": round(postot, 0),  # % of total positives identified
                    'PosPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=1), 3),  # % of positive predictions that were actually positive
                    "NegTot": round(negtot, 0),  # % of total negatives identified
                    'NegPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3),  # % of negative predictions that were actually negative
                    'PosCnt': sum(y_pred_valid == 1),
                    'NegCnt': sum(y_pred_valid == 0),
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [8, 10, 12], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }

    if date == 'lag':
        p = 350
        df_ind_rec = df_indicators.iloc[:p].copy()
        df_pred_rec = df_predict.iloc[:p].copy()
        df_indicators = df_indicators.iloc[p:]
        df_predict = df_predict.iloc[p:]

        
        # Split data once
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.10, random_state=42, shuffle=True)
        X_test = pd.concat([X_test, df_ind_rec], axis=0)
        y_test = pd.concat([y_test, df_pred_rec], axis=0)
        
        """
        X_test = df_indicators.iloc[:p].copy()
        y_test = df_predict.iloc[:p].copy()
        X_train = df_indicators.iloc[p:].copy()
        y_train = df_predict.iloc[p:].copy()
        
        print(f'xte {len(X_test)} | xtr{len(X_train)} | yte{len(y_test)} | ytr{len(y_train)}')
        """

    else:
        
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=None, shuffle=True)

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_test, y_train, y_test, opt, thresh)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
    
def optimize_ttv(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                metrics[t] = {
                    'TT_Len': len(df_indicators),
                    #'PosF1': round(f1_score(y_test_valid, y_pred_valid), 3),
                    #'NegF1': round((2*round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3)*round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3))/(round(recall_score(y_test_valid, y_pred_valid, pos_label=0), 3) + round(precision_score(y_test_valid, y_pred_valid, pos_label=0), 3)),3),
                    "PosTot": round(postot, 0),  # % of total positives identified
                    "NegTot": round(negtot, 0),  # % of total negatives identified
                    'PosPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 3),  # % of positive predictions that were actually positive
                    'NegPrec': round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 3),  # % of negative predictions that were actually negative
                    'PosCnt': sum(y_pred_valid == 1),
                    'NegCnt': sum(y_pred_valid == 0),
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400, 500],
            'max_depth': [14], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [5, 7],
            'early_stopping_rounds': [10]
        }

    if date == 'lag':
        
        p = 100
        X_test = df_indicators.iloc[:p].copy()
        y_test = df_predict.iloc[:p].copy()
        df_indicators = df_indicators.iloc[p:].copy()
        df_predict = df_predict.iloc[p:].copy()

        # Split data once
        #X_train, X_val, y_train, y_val = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=42, shuffle=False)
        
        p=250
        X_val = df_indicators.iloc[:p].copy()
        y_val = df_predict.iloc[:p].copy()
        X_train = df_indicators.iloc[p:].copy()
        y_train = df_predict.iloc[p:].copy()
        """
        print(f'xte {len(X_test)} | xtr{len(X_train)} | yte{len(y_test)} | ytr{len(y_train)}')
        """

    else:
        
        X_train, X_test, y_train, y_test = train_test_split(df_indicators, df_predict, test_size=0.3, random_state=None, shuffle=True)

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model
    
def optimize_ttv2(df_indicators, df_predict, thresh, opt, scale_pos_weight, arch, test_size, val_size, perf_size, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh):
        
        # Create a Stratified K-Fold object
        stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # Perform Random Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,  # Corrected from param_grid to param_distributions
            scoring=opt,
            cv=stratified_kfold,
            n_jobs=-1,
            n_iter=40,  # Adjust this based on how many random samples you want to try
            random_state=42  # Ensures reproducibility
        )

        random_search.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        verbose=False)
        best_model = random_search.best_estimator_
        # Predict probabilities
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        # Evaluate metrics for each threshold
        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
                poscnt = sum(y_pred_valid == 1)
                negcnt = sum(y_pred_valid == 0)
                posrec = round(posprec * poscnt / postot, 2)
                negrec = round(negprec * negcnt / negtot, 2)
                metrics[t] = {
                    'TT_Len': len(df_indicators),
                    "PosTot": postot,  
                    "NegTot": negtot,  
                    "PosRec": posrec,
                    'PosPrec': posprec,  
                    "NegRec": negrec,
                    'NegPrec': negprec,  
                    'PosFb': round((3*posprec*posrec)/((2*posrec)+(1*posprec)),2),
                    'NegFb': round((3*negprec*negrec)/((2*negrec)+(1*negprec)),2),
                }
            else:
                metrics[t] = {'F1': 0, 'BalAcc': 0, 'PosAcc': 0, 'NegAcc': 0, 'PosCnt': 0, 'NegCnt': 0}

        del random_search
        gc.collect()

        return metrics, best_model

    if arch == 'shallow':
        # Shallow
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300],
            'max_depth': [5, 7], 
            'learning_rate': [0.01],
            'subsample': [0.65],
            'colsample_bytree': [0.6], 
            'gamma': [0.2, 0.4],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [12, 15],
            'early_stopping_rounds': [10]
        }

    elif arch == 'moderate':

        # Moderate
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [300, 400],
            'max_depth': [7, 9], 
            'learning_rate': [0.01],
            'subsample': [0.65, .75],
            'colsample_bytree': [0.6, 0.7], 
            'gamma': [0.2, 0.3],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [9, 11],
            'early_stopping_rounds': [8]
        }

    else:

        # Deep
        xgboost_hyperparameters = {
            'scale_pos_weight': [scale_pos_weight],
            'n_estimators': [200, 300, 400],
            'max_depth': [6, 9, 11], 
            'learning_rate': [0.01],
            'subsample': [0.75, .85],
            'colsample_bytree': [0.75, 0.85], 
            'gamma': [0.1, 0.2],
            'alpha': [0.1, 1], 
            'lambda': [1, 2, 5], 
            'min_child_weight': [7, 9],
            'early_stopping_rounds': [10]
        }
        
    X_test = df_indicators.iloc[:perf_size].copy()
    y_test = df_predict.iloc[:perf_size].copy()
    df_indicators = df_indicators.iloc[test_size:].copy()
    df_predict = df_predict.iloc[test_size:].copy()

    X_val = df_indicators.iloc[:val_size].copy()
    y_val = df_predict.iloc[:val_size].copy()
    X_train = df_indicators.iloc[val_size:].copy()
    y_train = df_predict.iloc[val_size:].copy()

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, X_val, y_train, y_val, X_test, y_test, opt, thresh)
    
    metrics = list(xg_metrics.values())[0]
    pscore = metrics.get('PosFb', 0)
    nscore = metrics.get('NegFb', 0)
    print_metrics(xg_metrics)
    """
    if pscore >= .8 and nscore >= .8:
        print_metrics(xg_metrics)
    """
    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model

raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()

# By category × velocity
raw_duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_duration_slow_10 = [f"{col}_EMA10" for col in raw_duration_slow]
raw_duration_slow_25 = [f"{col}_EMA25" for col in raw_duration_slow]

raw_duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_duration_moderate_10 = [f"{col}_EMA10" for col in raw_duration_moderate]
raw_duration_moderate_25 = [f"{col}_EMA25" for col in raw_duration_moderate]

raw_duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
raw_duration_fast_10 = [f"{col}_EMA10" for col in raw_duration_fast]
raw_duration_fast_25 = [f"{col}_EMA25" for col in raw_duration_fast]

raw_trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_trend_slow_10 = [f"{col}_EMA10" for col in raw_trend_slow]
raw_trend_slow_25 = [f"{col}_EMA25" for col in raw_trend_slow]

raw_trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_trend_moderate_10 = [f"{col}_EMA10" for col in raw_trend_moderate]
raw_trend_moderate_25 = [f"{col}_EMA25" for col in raw_trend_moderate]

raw_trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
raw_trend_fast_10 = [f"{col}_EMA10" for col in raw_trend_fast]
raw_trend_fast_25 = [f"{col}_EMA25" for col in raw_trend_fast]

raw_trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_trend_ratio_slow_10 = [f"{col}_EMA10" for col in raw_trend_ratio_slow]
raw_trend_ratio_slow_25 = [f"{col}_EMA25" for col in raw_trend_ratio_slow]

raw_trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_trend_ratio_moderate_10 = [f"{col}_EMA10" for col in raw_trend_ratio_moderate]
raw_trend_ratio_moderate_25 = [f"{col}_EMA25" for col in raw_trend_ratio_moderate]

raw_trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
raw_trend_ratio_fast_10 = [f"{col}_EMA10" for col in raw_trend_ratio_fast]
raw_trend_ratio_fast_25 = [f"{col}_EMA25" for col in raw_trend_ratio_fast]

raw_volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_volatility_slow_10 = [f"{col}_EMA10" for col in raw_volatility_slow]
raw_volatility_slow_25 = [f"{col}_EMA25" for col in raw_volatility_slow]

raw_volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_volatility_moderate_10 = [f"{col}_EMA10" for col in raw_volatility_moderate]
raw_volatility_moderate_25 = [f"{col}_EMA25" for col in raw_volatility_moderate]

raw_volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
raw_volatility_fast_10 = [f"{col}_EMA10" for col in raw_volatility_fast]
raw_volatility_fast_25 = [f"{col}_EMA25" for col in raw_volatility_fast]

raw_momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
raw_momentum_slow_10 = [f"{col}_EMA10" for col in raw_momentum_slow]
raw_momentum_slow_25 = [f"{col}_EMA25" for col in raw_momentum_slow]

raw_momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
raw_momentum_moderate_10 = [f"{col}_EMA10" for col in raw_momentum_moderate]
raw_momentum_moderate_25 = [f"{col}_EMA25" for col in raw_momentum_moderate]

raw_momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
raw_momentum_fast_10 = [f"{col}_EMA10" for col in raw_momentum_fast]
raw_momentum_fast_25 = [f"{col}_EMA25" for col in raw_momentum_fast]

# Category based groupings

In [134]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.55]
arch_types = ['deep', 'moderate', 'shallow']
dates = ['lag']#, 'lag']
n = len(raw_cols_list)
results = []

for ticker in tickers:
    
    if ticker == 'QQQ':
        dates = ['lag']#, 'current']
    else:
        dates = ['lag']

    for date in dates:

        if ticker == 'QQQ':
                returns = [5, 10, 20, 30]#[4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
        else:
            returns = [4, 6, 8, 10, 15]

        for r in returns:

            # loop r = 1 .. 5
            for i in range(1, n+1):
                
                # all index-combos of length r
                for idx_combo in itertools.combinations(range(n), i):

                    # pull out the names and the column‐lists
                    name_combo = [ raw_names[i]    for i in idx_combo ]
                    cols_combo = [ raw_cols_list[i] for i in idx_combo ]
                    
                    # flatten the list of lists into one feature list
                    cols = [ feat for sub in cols_combo for feat in sub ]
                    
                    # build a descriptive name, e.g. "trend_volatility_momentum"
                    combo_name = "_".join(name_combo)

                    df_ph = df.copy()
                    df_ph = df_ph.iloc[r:].copy()
                    return_col = f"Return_{r}"
                    model_key = f"QQQ_{r}"
                    counts = df_ph[return_col].value_counts()
                    neg = counts.get(0, 0)
                    pos = counts.get(1, 1)  # prevent division by zero
                    scale_pos_weight = neg / pos
                    imbalance_ratio = min(pos, neg) / max(pos, neg)
                    min_child_weight = max(int(max(1, round(10 * imbalance_ratio))),5)

                    for arch in arch_types:

                        depth = min(int(math.floor(len(cols) / 2)), 10)
                        depth = min(int(round(len(cols) / 2, 0)), 10)
                        depth = max(depth, 6) #minimum depth set to 6 if below 6

                        # Choose evaluation metric
                        opt = 'matthews_corrcoef'

                        # Combine features with return column, drop missing
                        #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                        used_cols = cols + [return_col]
                        df_model = df_ph[used_cols].dropna()
                        #print(df_ph['Date'].iloc[0])
                        
                        df_indicators = df_model[cols]
                        df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                        df_predict = df_model[return_col]
                        
                        print(f"Results for {combo_name} | {arch} | {ticker}_{r} | {date}")
                        xg_metrics, best_xg_model = optimize_tests(df_indicators, df_predict, thresh, opt, depth, scale_pos_weight, min_child_weight, r, name, arch, date, return_metrics=True)
                        metrics = list(xg_metrics.values())[0]
                        # build one flat record
                        row = {'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'date': date, **metrics}
                        results.append(row)
                        print('---------------------------')

Results for duration | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.774, 'NegF1': 0.618, "PosID'd": 0.826, 'PosPrec': 0.728, "NegID'd": 0.558, 'NegPrec': 0.692, 'PosCnt': 294, 'NegCnt': 146}
---------------------------
Results for duration | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.743, 'NegF1': 0.576, "PosID'd": 0.771, 'PosPrec': 0.717, "NegID'd": 0.544, 'NegPrec': 0.613, 'PosCnt': 272, 'NegCnt': 150}
---------------------------
Results for duration | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.709, 'NegF1': 0.552, "PosID'd": 0.718, 'PosPrec': 0.7, "NegID'd": 0.541, 'NegPrec': 0.563, 'PosCnt': 240, 'NegCnt': 151}
---------------------------
Results for trend | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.716, 'NegF1': 0.549, "PosID'd": 0.719, 'PosPrec': 0.713, "NegID'd": 0.545, 'NegPrec': 0.553, 'PosCnt': 244, 'NegCnt': 152}
---------------------------
Results for trend | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.731, 'NegF1': 0.588, "PosID'd": 0

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.731, 'NegF1': 0.52, "PosID'd": 0.777, 'PosPrec': 0.689, "NegID'd": 0.47, 'NegPrec': 0.582, 'PosCnt': 309, 'NegCnt': 146}
---------------------------
Results for trend_momentum | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.718, 'NegF1': 0.52, "PosID'd": 0.757, 'PosPrec': 0.683, "NegID'd": 0.478, 'NegPrec': 0.57, 'PosCnt': 224, 'NegCnt': 114}
---------------------------
Results for trend_momentum | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.728, 'NegF1': 0.566, "PosID'd": 0.79, 'PosPrec': 0.675, "NegID'd": 0.503, 'NegPrec': 0.648, 'PosCnt': 240, 'NegCnt': 122}
---------------------------
Results for trend_ratio_volatility | deep | QQQ_5 | lag


/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.724, 'NegF1': 0.528, "PosID'd": 0.774, 'PosPrec': 0.68, "NegID'd": 0.476, 'NegPrec': 0.594, 'PosCnt': 272, 'NegCnt': 133}
---------------------------
Results for trend_ratio_volatility | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.716, 'NegF1': 0.538, "PosID'd": 0.726, 'PosPrec': 0.706, "NegID'd": 0.526, 'NegPrec': 0.55, 'PosCnt': 218, 'NegCnt': 129}
---------------------------
Results for trend_ratio_volatility | shallow | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.687, 'NegF1': 0.513, "PosID'd": 0.699, 'PosPrec': 0.676, "NegID'd": 0.5, 'NegPrec': 0.527, 'PosCnt': 182, 'NegCnt': 112}
---------------------------
Results for trend_ratio_momentum | deep | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.74, 'NegF1': 0.539, "PosID'd": 0.788, 'PosPrec': 0.697, "NegID'd": 0.486, 'NegPrec': 0.605, 'PosCnt': 310, 'NegCnt': 147}
---------------------------
Results for trend_ratio_momentum | moderate | QQQ_5 | lag
  Threshold 0.55: {'PosF1': 0.744, 'NegF1': 0.552,

...

  Threshold 0.55: {'PosF1': 0.737, 'NegF1': 0.597, "PosID'd": 0.708, 'PosPrec': 0.768, "NegID'd": 0.637, 'NegPrec': 0.562, 'PosCnt': 246, 'NegCnt': 178}
---------------------------
Results for duration_trend_ratio_volatility | shallow | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.735, 'NegF1': 0.561, "PosID'd": 0.685, 'PosPrec': 0.794, "NegID'd": 0.64, 'NegPrec': 0.5, 'PosCnt': 238, 'NegCnt': 174}
---------------------------
Results for duration_trend_ratio_momentum | deep | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.78, 'NegF1': 0.65, "PosID'd": 0.743, 'PosPrec': 0.821, "NegID'd": 0.706, 'NegPrec': 0.602, 'PosCnt': 296, 'NegCnt': 211}
---------------------------
Results for duration_trend_ratio_momentum | moderate | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.746, 'NegF1': 0.613, "PosID'd": 0.689, 'PosPrec': 0.813, "NegID'd": 0.701, 'NegPrec': 0.545, 'PosCnt': 262, 'NegCnt': 211}
---------------------------
Results for duration_trend_ratio_momentum | shallow | QQQ_10 | lag
  Threshold 0

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.751, 'NegF1': 0.548, "PosID'd": 0.711, 'PosPrec': 0.796, "NegID'd": 0.611, 'NegPrec': 0.497, 'PosCnt': 284, 'NegCnt': 183}
---------------------------
Results for trend_trend_ratio_volatility | moderate | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.722, 'NegF1': 0.572, "PosID'd": 0.679, 'PosPrec': 0.77, "NegID'd": 0.633, 'NegPrec': 0.522, 'PosCnt': 239, 'NegCnt': 182}
---------------------------
Results for trend_trend_ratio_volatility | shallow | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.724, 'NegF1': 0.539, "PosID'd": 0.679, 'PosPrec': 0.775, "NegID'd": 0.606, 'NegPrec': 0.485, 'PosCnt': 240, 'NegCnt': 171}
---------------------------
Results for trend_trend_ratio_momentum | deep | QQQ_10 | lag
  Threshold 0.55: {'PosF1': 0.776, 'NegF1': 0.633, "PosID'd": 0.729, 'PosPrec': 0.83, "NegID'd": 0.708, 'NegPrec': 0.573, 'PosCnt': 276, 'NegCnt': 199}
---------------------------
Results for trend_trend_ratio_momentum | moderate | QQQ_10 | lag
  Threshold 0.55: {

/Users/brettchase/Library/Python/3.12/lib/python/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Threshold 0.55: {'PosF1': 0.781, 'NegF1': 0.599, "PosID'd": 0.72, 'PosPrec': 0.852, "NegID'd": 0.708, 'NegPrec': 0.519, 'PosCnt': 305, 'NegCnt': 210}
---------------------------
Results for duration_trend_trend_ratio_volatility_momentum | shallow | QQQ_20 | lag
  Threshold 0.55: {'PosF1': 0.776, 'NegF1': 0.632, "PosID'd": 0.706, 'PosPrec': 0.861, "NegID'd": 0.755, 'NegPrec': 0.544, 'PosCnt': 273, 'NegCnt': 215}
---------------------------
Results for duration | deep | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.814, 'NegF1': 0.525, "PosID'd": 0.842, 'PosPrec': 0.788, "NegID'd": 0.483, 'NegPrec': 0.574, 'PosCnt': 353, 'NegCnt': 122}
---------------------------
Results for duration | moderate | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.76, 'NegF1': 0.47, "PosID'd": 0.769, 'PosPrec': 0.752, "NegID'd": 0.458, 'NegPrec': 0.482, 'PosCnt': 314, 'NegCnt': 137}
---------------------------
Results for duration | shallow | QQQ_30 | lag
  Threshold 0.55: {'PosF1': 0.758, 'NegF1': 0.481, "PosID'd

# Corrected Timesplit Code

In [11]:
categories = ['lag', 'duration', 'trend', 'trend_ratio', 'volatility', 'momentum']
velocities = ['slow', 'moderate', 'fast']
suffix_windows = ['EMA10'] #['EMA10', 'EMA25']

raw_features = {}

for cat in categories:

    for vel in velocities:
        
        base_key = f'raw_{cat}_{vel}'
        indicators = tags[(tags['Type'] == 'Raw') & 
                          (tags['Category'] == cat) & 
                          (tags['velocity_25_cat_local'] == vel)]['Indicator'].tolist()
        
        # Store base list
        raw_features[base_key] = indicators
        
        # Only apply suffixes to specific categories/velocities if needed
        if cat in categories and vel in velocities:

            if cat in ('volatility', 'momentum') and vel == 'fast':
                
                for suffix in suffix_windows:

                    raw_features[f'{base_key}_{suffix}'] = [f"{col}_{suffix}" for col in indicators]

# Add in seasonality
weekly = ['cyc_week_sin', 'cyc_week_cos']
monthly = ['cyc_month_sin', 'cyc_month_cos']
quarterly = ['cyc_quarter_sin', 'cyc_quarter_cos']
yearly = ['cyc_year_sin', 'cyc_year_cos']
all_seasons = ['cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos',
               'cyc_year_sin', 'cyc_year_cos']
"""
raw_features['weekly'] = weekly
raw_features['monthly'] = monthly
raw_features['quarterly'] = quarterly
raw_features['yearly'] = yearly
"""
raw_features['all_seasons'] = all_seasons

def base_name(name):

    suffixes = ("_EMA10", "_EMA25")

    for s in suffixes:
        if name.endswith(s):
            return name[: -len(s)]
    return name

# 1) Map your 15 groups to their feature‐lists
group_cols = {
    key: val for key, val in raw_features.items()
    if len(val) > 0  # optional: only include non-empty lists
}

group_names = list(group_cols.keys())

# 2) Generate all 5‐combos, filtering out any with >2 from one category
valid_combos = []
for combo in itertools.combinations(group_names, 3):
    # normalize names by removing EMA suffixes
    bases = [base_name(name) for name in combo]
    # skip if both collapse to the same base
    if bases[0] == bases[1]:
        continue
    # (your category count rule, if still needed)
    if all(bases.count(cat) <= 2 for cat in set(bases)):
        valid_combos.append(combo)

# 3) (Optional) Build a mapping from combo → flattened feature list
combos_features = {
    combo: [feat for grp in combo for feat in group_cols[grp]]
    for combo in valid_combos
}

print(len(combos_features))

963


In [13]:
def optimize_splits_new(df_indicators, df_predict, thresh, opt, arch, test_size, withold, return_metrics=False):
    
    def train_and_evaluate(model, param_grid, X_train, y_train, X_test, y_test, opt, thresh, iter):

        model.set_params(n_jobs=1)
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            scoring=opt,
            cv=5,
            n_jobs=-1,
            n_iter=iter,
            random_state=42
        )
        random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        best_model = random_search.best_estimator_
        y_prob = best_model.predict_proba(X_test)
        postot = y_test.sum()
        negtot = len(y_test) - y_test.sum()

        metrics = {}
        for t in thresh:
            y_pred_thresh = (y_prob[:, 1] > t).astype(int)
            y_pred_thresh[y_prob[:, 0] > t] = 0
            filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

            if filtered_indices.sum() > 0:
                y_test_valid = y_test[filtered_indices]
                y_pred_valid = y_pred_thresh[filtered_indices]
                posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
                negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
                poscnt = sum(y_pred_valid == 1)
                negcnt = sum(y_pred_valid == 0)
                posrec = round(posprec * poscnt / postot, 2)
                negrec = round(negprec * negcnt / negtot, 2)
                metrics[t] = {
                    "Len": len(y_test),
                    "PC": poscnt,
                    "NC": negcnt,
                    "PP": posprec,
                    "PR": posrec,
                    "NP": negprec,
                    "NR": negrec,
                    "PosFb": round((3*posprec*posrec)/((2*posrec)+(1*posprec)),3),
                    "NegFb": round((3*negprec*negrec)/((2*negrec)+(1*negprec)),3),
                }

        return metrics, best_model

    X_test = df_indicators.iloc[:test_size].copy()
    y_test = df_predict.iloc[:test_size].copy()
    
    if withold == 'N':
        test_size = 0
    df_indicators = df_indicators.iloc[test_size:].sort_index(ascending=True).copy()
    df_predict = df_predict.iloc[test_size:].sort_index(ascending=True).copy()

    #X_val = df_indicators.iloc[:val_size].copy()
    #y_val = df_predict.iloc[:val_size].copy()
    X_train = df_indicators.copy()
    y_train = df_predict.copy()

    counts = y_train.value_counts()
    negf = counts.get(0, 0)
    posf = counts.get(1, 1)  # prevent division by zero
    scale_pos_weight = negf / posf

    if arch == 'deep':

        # Deep
        xgboost_hyperparameters = {
            'max_depth': [6, 9, 12],          # Still fairly deep
            'learning_rate': [0.01, 0.05],    # Lower rates
            'min_child_weight': [4, 7, 9, 12],   # More conservative splits
            'subsample': [0.7, 0.8, 0.9],     
            'colsample_bytree': [0.75, 0.85], 
            'colsample_bylevel': [0.7, 0.8, 0.9],
            'colsample_bynode': [0.7, 0.8, 0.9],    
            'gamma': [0.1, 0.3, 0.5],         
            'alpha': [0.1, 0.5, 1.0],         
            'lambda': [5, 10, 15],            
            'n_estimators': [400, 600, 800, 1000], 
            'early_stopping_rounds': [10]
            }

    iter = 50

    # Train and evaluate models
    xg_metrics, best_xg_model = train_and_evaluate(XGBClassifier(random_state=42), xgboost_hyperparameters, X_train, y_train, X_test, y_test, opt, thresh, iter)
    #record_validation_metrics(xg_metrics, arch='shallow', horizon=r, model_name=name)
    #savearch(best_xg_model, r, name, arch)
    metrics = list(xg_metrics.values())[0]
    pscore = metrics.get('PosFb', 0)
    nscore = metrics.get('NegFb', 0)

    if pscore > .75 and nscore > .65:
        print_metrics(xg_metrics)

    if return_metrics:
        return xg_metrics, best_xg_model
    else:
        #print_metrics(xg_metrics)
        return best_xg_model

def add_cyclic_seasonality(
    df: pd.DataFrame,
    date_col: str = 'Date',
    add_weekly: bool = True,
    add_month: bool = True,
    add_quarter: bool = True,
    add_year: bool = True,
    add_day_of_year: bool = False,
    mode: str = 'calendar',          # 'trading', 'calendar', or 'both'
    prefix: str = 'cyc_'
) -> pd.DataFrame:
    """
    Add cyclic (sin/cos) seasonality features.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a date column.
    date_col : str
        Name of the datetime column.
    add_weekly, add_month, add_quarter, add_year, add_day_of_year : bool
        Toggles for which seasonal cycles to create.
    mode : {'trading','calendar','both'}
        - 'trading': position = trading-day index / trading-days-in-period (resets at dataset slice start)
        - 'calendar': position = calendar progress within period (day-based; retains true mid-period phase)
        - 'both': create both sets (with suffixes _trade and _cal)
    prefix : str
        Prefix for created feature names.

    Returns
    -------
    pd.DataFrame
        Copy of df with new seasonality features appended (original row order preserved).
    """
    if date_col not in df.columns:
        raise ValueError(f"'{date_col}' not found in DataFrame.")
    if mode not in {'trading', 'calendar', 'both'}:
        raise ValueError("mode must be 'trading', 'calendar', or 'both'.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Preserve original order / index
    original_index = out.index
    work = out.sort_values(date_col).reset_index(drop=False)
    idx_col = work.columns[0]

    # Precompute basic date parts
    work['_year']    = work[date_col].dt.year
    work['_month']   = work[date_col].dt.month
    work['_quarter'] = work[date_col].dt.quarter
    work['_weekday'] = work[date_col].dt.weekday
    work['_doy']     = work[date_col].dt.dayofyear
    work['_is_leap'] = work[date_col].dt.is_leap_year

    # ---------- WEEKLY (trading week: Mon-Fri) ----------
    if add_weekly:
        week_pos = work['_weekday'] / 5.0  # Monday=0.0, Friday≈0.8 (if only Mon-Fri present)
        work[f'{prefix}week_sin'] = np.sin(2 * np.pi * week_pos)
        work[f'{prefix}week_cos'] = np.cos(2 * np.pi * week_pos)

    # Helper to safely divide
    def _safe_div(num, den):
        return np.where(den == 0, 0.0, num / np.where(den == 0, 1, den))

    # ---------- MONTH ----------
    if add_month:
        if mode in {'trading', 'both'}:
            grp_m = [work['_year'], work['_month']]
            idx_m = work.groupby(grp_m).cumcount()
            cnt_m = work.groupby(grp_m)[date_col].transform('count')
            pos_m_trade = _safe_div(idx_m, cnt_m)
            work[f'{prefix}month_trade_sin'] = np.sin(2 * np.pi * pos_m_trade)
            work[f'{prefix}month_trade_cos'] = np.cos(2 * np.pi * pos_m_trade)

        if mode in {'calendar', 'both'}:
            dom = work[date_col].dt.day           # 1..days_in_month
            dim = work[date_col].dt.days_in_month
            pos_m_cal = _safe_div(dom - 1, dim - 1)
            work[f'{prefix}month_sin'] = np.sin(2 * np.pi * pos_m_cal)
            work[f'{prefix}month_cos'] = np.cos(2 * np.pi * pos_m_cal)

    # ---------- QUARTER ----------
    if add_quarter:
        if mode in {'trading', 'both'}:
            grp_q = [work['_year'], work['_quarter']]
            idx_q = work.groupby(grp_q).cumcount()
            cnt_q = work.groupby(grp_q)[date_col].transform('count')
            pos_q_trade = _safe_div(idx_q, cnt_q)
            work[f'{prefix}quarter_trade_sin'] = np.sin(2 * np.pi * pos_q_trade)
            work[f'{prefix}quarter_trade_cos'] = np.cos(2 * np.pi * pos_q_trade)

        if mode in {'calendar', 'both'}:
            q = work['_quarter']
            y = work['_year']
            quarter_start = pd.to_datetime({'year': y, 'month': (q - 1) * 3 + 1, 'day': 1})
            quarter_end = quarter_start + pd.offsets.QuarterEnd(0)
            day_in_q = (work[date_col] - quarter_start).dt.days
            days_q = (quarter_end - quarter_start).dt.days
            pos_q_cal = _safe_div(day_in_q, days_q)
            work[f'{prefix}quarter_sin'] = np.sin(2 * np.pi * pos_q_cal)
            work[f'{prefix}quarter_cos'] = np.cos(2 * np.pi * pos_q_cal)

    # ---------- YEAR ----------
    if add_year:
        if mode in {'trading', 'both'}:
            grp_y = work['_year']
            idx_y = work.groupby(grp_y).cumcount()
            cnt_y = work.groupby(grp_y)[date_col].transform('count')
            pos_y_trade = _safe_div(idx_y, cnt_y)
            work[f'{prefix}year_trade_sin'] = np.sin(2 * np.pi * pos_y_trade)
            work[f'{prefix}year_trade_cos'] = np.cos(2 * np.pi * pos_y_trade)

        if mode in {'calendar', 'both'}:
            doy = work['_doy']
            year_len = np.where(work['_is_leap'], 366, 365)
            pos_y_cal = _safe_div(doy - 1, year_len - 1)
            work[f'{prefix}year_sin'] = np.sin(2 * np.pi * pos_y_cal)
            work[f'{prefix}year_cos'] = np.cos(2 * np.pi * pos_y_cal)

    # ---------- DAY OF YEAR (explicit) ----------
    if add_day_of_year:
        year_len = np.where(work['_is_leap'], 366, 365)
        work[f'{prefix}doy_sin'] = np.sin(2 * np.pi * work['_doy'] / year_len)
        work[f'{prefix}doy_cos'] = np.cos(2 * np.pi * work['_doy'] / year_len)

    # Drop temp columns
    temp_cols = [c for c in work.columns if c.startswith('_')]
    work = work.drop(columns=temp_cols)

    # Identify newly created features
    new_cols = [c for c in work.columns if c not in out.columns and c != idx_col]

    # Map back to original order
    work = work.set_index(idx_col)
    out[new_cols] = work.loc[original_index, new_cols]

    return out

weekly = ['cyc_week_sin', 'cyc_week_cos']
monthly = ['cyc_month_sin', 'cyc_month_cos']
quarterly = ['cyc_quarter_sin', 'cyc_quarter_cos']
yearly = ['cyc_year_sin', 'cyc_year_cos']
all_seasons = ['cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos',
               'cyc_year_sin', 'cyc_year_cos']

In [43]:
tickers = ['QQQ']
thresh = [.5]
#results = []
returns = [35]#[5, 10, 15, 20, 25, 35, 45]
#all_perm_dfs = []
arch_types = ['deep']
lb = 8
withold = 'Y'
test_size = 130

name_to_combo = { " + ".join(c): c for c in valid_combos }
raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()

for ticker in tickers:

    df = extract(ticker, returns, lb, raw_all, windows=[10,25])
    df = add_cyclic_seasonality(df, date_col='Date', add_weekly=True, add_month=True,
                                add_quarter=True, add_year=True, add_day_of_year=False,
                                mode='calendar',    # or 'trading' or 'both'
                                prefix='cyc_')    

    for r in returns:
            
        for combo in valid_combos:
            
            combo = ('raw_lag_fast', 'raw_trend_fast', 'raw_trend_ratio_moderate')
            combo_name = "+".join(combo)
            cols = combos_features[combo]
            
            df_ph = df.copy()
            df_ph = df_ph.iloc[r:].copy()
            return_col = f"Return_{r}"
            return_perc_col = f"Return%_{r}"
            model_key = f"QQQ_{r}"

            rets = df_ph[return_perc_col].copy()
            # Split into negative and positive returns
            neg = rets[rets < 0]
            pos = rets[rets > 0]
            # Calculate dynamic thresholds
            neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
            pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude
            
            filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()

            for arch in arch_types:

                # Choose evaluation metric
                opt = 'matthews_corrcoef'

                # Combine features with return column, drop missing
                #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                cols = list(dict.fromkeys(cols))
                used_cols = cols + [return_col]
                df_model = filtered[used_cols].dropna()
                
                df_indicators = df_model[cols]
                df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                df_predict = df_model[return_col]
                
                print(f"Results for {combo_name} | {arch} | {ticker}_{r}")
                xg_metrics, best_xg_model = optimize_splits_new(df_indicators, df_predict, thresh, opt, arch, test_size, withold, return_metrics=True)
                metrics = list(xg_metrics.values())[0]
                pscore = metrics.get('PosFb', 0)
                nscore = metrics.get('NegFb', 0)
                if pscore > .75 and nscore > .65:
                    row = {'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, **metrics}
                    results.append(row)
                
                print('---------------------------')

Results for raw_lag_fast+raw_trend_fast+raw_trend_ratio_moderate | deep | QQQ_35
  Threshold 0.5: {'Len': 130, 'PC': 104, 'NC': 26, 'PP': 0.99, 'PR': 1.0, 'NP': 1.0, 'NR': 0.96, 'PosFb': 0.993, 'NegFb': 0.986}
---------------------------
Results for raw_lag_fast+raw_trend_fast+raw_trend_ratio_moderate | deep | QQQ_35
  Threshold 0.5: {'Len': 130, 'PC': 104, 'NC': 26, 'PP': 0.99, 'PR': 1.0, 'NP': 1.0, 'NR': 0.96, 'PosFb': 0.993, 'NegFb': 0.986}
---------------------------
Results for raw_lag_fast+raw_trend_fast+raw_trend_ratio_moderate | deep | QQQ_35
  Threshold 0.5: {'Len': 130, 'PC': 104, 'NC': 26, 'PP': 0.99, 'PR': 1.0, 'NP': 1.0, 'NR': 0.96, 'PosFb': 0.993, 'NegFb': 0.986}
---------------------------
Results for raw_lag_fast+raw_trend_fast+raw_trend_ratio_moderate | deep | QQQ_35
  Threshold 0.5: {'Len': 130, 'PC': 104, 'NC': 26, 'PP': 0.99, 'PR': 1.0, 'NP': 1.0, 'NR': 0.96, 'PosFb': 0.993, 'NegFb': 0.986}
---------------------------
Results for raw_lag_fast+raw_trend_fast+raw_tren

KeyboardInterrupt: 

In [45]:
df_indicators

,QQQ_SMA_10_Lag50,QQQ_SMA_25_Lag50,QQQ_SMA_50_Lag50,QQQ_SMA_100_Lag50,QQQ_SMA_200_Lag50,QQQ_SMA_10_Lag100,QQQ_SMA_25_Lag100,QQQ_SMA_50_Lag100,QQQ_SMA_100_Lag100,QQQ_SMA_200_Lag100,...,QQQ_SMA_200_Lag200,minus_DI,plus_DI,10_EMA_25,25_SMA_100,10_EMA_50,25_SMA_50,25_EMA_200,25_EMA_100,25_SMA_200
1975,1.09,1.14,1.15,1.03,1.04,1.17,1.10,1.07,1.00,1.04,...,1.13,19.0,33.0,1.010,1.096,1.033,1.026,1.096,1.057,1.097
1974,1.09,1.14,1.15,1.03,1.04,1.16,1.10,1.06,1.00,1.04,...,1.13,21.0,30.0,1.010,1.096,1.033,1.026,1.096,1.057,1.097
1973,1.10,1.15,1.15,1.03,1.04,1.15,1.09,1.06,1.00,1.04,...,1.13,24.0,31.0,1.009,1.097,1.033,1.027,1.095,1.057,1.096
1972,1.10,1.15,1.15,1.03,1.04,1.14,1.09,1.05,1.00,1.04,...,1.14,25.0,30.0,1.011,1.098,1.035,1.028,1.096,1.058,1.096
1971,1.11,1.16,1.14,1.03,1.04,1.14,1.08,1.05,0.99,1.04,...,1.14,29.0,28.0,1.011,1.099,1.036,1.029,1.096,1.058,1.095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,1.09,1.08,1.06,1.03,0.00,1.08,1.07,1.07,1.08,0.00,...,0.00,24.0,28.0,1.012,1.043,1.028,1.013,1.076,1.039,1.081
203,1.10,1.09,1.06,1.03,0.00,1.09,1.07,1.07,1.08,0.00,...,0.00,26.0,29.0,1.013,1.043,1.029,1.014,1.076,1.039,1.082
202,1.10,1.09,1.06,1.03,0.00,1.09,1.07,1.07,1.08,0.00,...,0.00,24.0,30.0,1.013,1.043,1.029,1.015,1.075,1.039,1.082
201,1.10,1.09,1.05,1.03,0.00,1.10,1.07,1.07,1.08,0.00,...,0.00,23.0,29.0,1.013,1.043,1.028,1.016,1.074,1.038,1.082


In [46]:
df_ph

,Date,Close,High,Low,Volume,QQQ_SMA_10,QQQ_EMA_10,QQQ_SMA_25,QQQ_EMA_25,QQQ_SMA_50,...,QQQ_SMA_100_Lag200_EMA25,QQQ_SMA_200_Lag200_EMA25,cyc_week_sin,cyc_week_cos,cyc_month_sin,cyc_month_cos,cyc_quarter_sin,cyc_quarter_cos,cyc_year_sin,cyc_year_cos
1975,2025-08-07,568.580872,572.656164,564.455649,44463000,1.008,1.009,1.015,1.019,1.041,...,1.082229,1.138550,-0.587785,-0.809017,9.510565e-01,0.309017,0.553775,-0.832666,-0.582185,-0.813056
1974,2025-08-06,566.663086,567.102579,559.980830,41823700,1.005,1.007,1.013,1.017,1.039,...,1.081581,1.139263,0.587785,-0.809017,8.660254e-01,0.500000,0.609902,-0.792477,-0.568065,-0.822984
1973,2025-08-05,559.621277,565.903972,559.081863,48666600,0.993,0.996,1.002,1.006,1.029,...,1.081713,1.140035,0.951057,0.309017,7.431448e-01,0.669131,0.663123,-0.748511,-0.553775,-0.832666
1972,2025-08-04,563.446777,563.666554,558.302777,47669800,1.000,1.002,1.009,1.013,1.038,...,1.081856,1.140871,0.000000,1.000000,5.877853e-01,0.809017,0.713183,-0.700978,-0.539320,-0.842101
1971,2025-08-01,553.238647,558.372710,551.041183,69400800,0.982,0.985,0.992,0.996,1.021,...,1.082010,1.140943,-0.951057,0.309017,0.000000e+00,1.000000,0.842101,-0.539320,-0.495009,-0.868888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,2018-07-20,170.726105,171.851618,170.535327,35897300,1.001,1.004,1.019,1.016,1.032,...,0.000000,0.000000,-0.951057,0.309017,-7.431448e-01,-0.669131,0.966666,0.256039,-0.305732,-0.952118
203,2018-07-19,170.764252,171.575013,170.563955,31094100,1.004,1.005,1.019,1.018,1.034,...,0.000000,0.000000,-0.587785,-0.809017,-5.877853e-01,-0.809017,0.946699,0.322120,-0.289252,-0.957253
202,2018-07-18,171.622681,172.109143,171.098086,23220800,1.012,1.011,1.025,1.024,1.041,...,0.000000,0.000000,0.587785,-0.809017,-4.067366e-01,-0.913545,0.922220,0.386667,-0.272686,-0.962103
201,2018-07-17,171.946991,172.309437,169.095033,31034100,1.019,1.016,1.028,1.029,1.044,...,0.000000,0.000000,0.951057,0.309017,-2.079117e-01,-0.978148,0.893346,0.449370,-0.256039,-0.966666


In [16]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'Results.csv')

In [74]:
import pandas as pd

# ------------------------------------------------------------------
# 1. Build atomic feature lists from tags
# ------------------------------------------------------------------
def build_raw_features(tags: pd.DataFrame):
    categories = ['duration', 'trend', 'trend_ratio', 'volatility', 'momentum']
    velocities = ['slow', 'moderate', 'fast']
    suffix_windows = ['EMA10', 'EMA25']
    
    raw_features = {}
    
    for cat in categories:
        for vel in velocities:
            base_key = f'raw_{cat}_{vel}'
            indicators = tags[
                (tags['Type'] == 'Raw') &
                (tags['Category'] == cat) &
                (tags['velocity_25_cat_local'] == vel)
            ]['Indicator'].tolist()
            
            # Base list
            raw_features[base_key] = indicators
            
            # Suffix variants
            for suffix in suffix_windows:
                raw_features[f'{base_key}_{suffix}'] = [f"{col}_{suffix}" for col in indicators]
    
    return raw_features

# Build once (assumes 'tags' exists)
raw_features = build_raw_features(tags)

# ------------------------------------------------------------------
# 2. Selected base PAIRS per horizon (store as name tuples)
#    Replace / extend exactly with your current selected pairs
# ------------------------------------------------------------------
selected_pairs = {
    "h_2": [
        ("raw_trend_fast", "raw_trend_ratio_slow_EMA25"),
        ("raw_duration_slow_EMA10", "raw_trend_moderate"),
        ("raw_trend_ratio_moderate_EMA25", "raw_volatility_moderate_EMA25"),
        ("raw_trend_ratio_fast_EMA25", "raw_momentum_moderate_EMA10"),
        ("raw_trend_ratio_moderate_EMA10", "raw_momentum_moderate_EMA25"),
    ],
    "h_3": [
        ("raw_trend_ratio_moderate_EMA25", "raw_volatility_moderate_EMA25"),
        ("raw_trend_fast", "raw_trend_ratio_slow_EMA25"),
        ("raw_duration_fast", "raw_trend_slow"),
        ("raw_trend_ratio_slow", "raw_momentum_slow"),
        ("raw_trend_ratio_fast_EMA25", "raw_momentum_moderate_EMA25"),
    ],
    "h_4": [
        ("raw_trend_slow", "raw_trend_ratio_fast_EMA25"),
        ("raw_trend_slow", "raw_trend_ratio_moderate_EMA10"),
        ("raw_trend_fast", "raw_trend_ratio_moderate_EMA25"),
        ("raw_duration_moderate_EMA10", "raw_volatility_moderate_EMA25"),
        ("raw_trend_slow_EMA25", "raw_momentum_moderate"),
    ],
    "h_5": [
        ("raw_duration_slow_EMA10", "raw_trend_ratio_fast_EMA25"),
        ("raw_duration_slow", "raw_trend_ratio_fast_EMA25"),
        ("raw_trend_slow", "raw_trend_ratio_slow"),
        ("raw_trend_slow_EMA25", "raw_momentum_moderate"),
        ("raw_trend_moderate_EMA25", "raw_trend_ratio_moderate_EMA25"),
    ],
    "h_8": [
        ("raw_trend_fast", "raw_trend_ratio_moderate"),
        ("raw_trend_fast_EMA10", "raw_trend_ratio_slow_EMA25"),
        ("raw_trend_ratio_slow_EMA10", "raw_momentum_moderate"),
        ("raw_volatility_moderate_EMA25", "raw_momentum_slow_EMA25"),
        ("raw_trend_slow", "raw_trend_slow_EMA25"),
    ],
    "h_10": [
        ("raw_trend_ratio_slow_EMA25", "raw_volatility_moderate_EMA25"),
        ("raw_duration_moderate", "raw_trend_slow"),
        ("raw_duration_moderate_EMA25", "raw_trend_slow_EMA10"),
        ("raw_trend_ratio_moderate", "raw_trend_ratio_fast"),
        ("raw_trend_ratio_slow_EMA10", "raw_momentum_moderate_EMA10"),
    ],
    "h_15": [
        ("raw_trend_slow_EMA10", "raw_trend_ratio_moderate_EMA25"),
        ("raw_duration_moderate", "raw_trend_slow_EMA25"),
        ("raw_trend_ratio_moderate_EMA10", "raw_momentum_moderate_EMA25"),
        ("raw_trend_ratio_fast", "raw_momentum_moderate"),
        ("raw_trend_ratio_fast_EMA10", "raw_volatility_moderate"),
    ],
    "h_20": [
        ("raw_duration_slow", "raw_momentum_moderate_EMA25"),
        ("raw_trend_ratio_moderate_EMA25", "raw_momentum_moderate_EMA25"),
        ("raw_trend_slow_EMA25", "raw_momentum_moderate_EMA10"),
        ("raw_trend_ratio_moderate", "raw_volatility_moderate"),
        ("raw_trend_moderate", "raw_trend_ratio_fast_EMA25"),
    ],
}

# ------------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------------
def base_name(key: str) -> str:
    """Strip EMA suffix if present."""
    if key.endswith('_EMA10') or key.endswith('_EMA25'):
        return key.rsplit('_', 1)[0]
    return key

def expand_pair(a_name: str, b_name: str, lookup: dict) -> list:
    """Concatenate two feature lists and dedupe preserving order."""
    merged = lookup[a_name] + lookup[b_name]
    return list(dict.fromkeys(merged))

def pair_sets_for_horizon(horizon: int):
    h_key = f"h_{horizon}"
    if h_key not in selected_pairs:
        raise ValueError(f"Unsupported horizon {horizon}")
    return {
        f"{a}+{b}": expand_pair(a, b, raw_features)
        for a, b in selected_pairs[h_key]
    }

def build_triples_for_pairs(pair_list, lookup):
    """Given list of (a,b) pairs, build all triples a+b+c with allowed c."""
    all_keys = list(lookup.keys())
    triple_map = {}
    for a, b in pair_list:
        base_a, base_b = base_name(a), base_name(b)
        excluded = {base_a, base_b}
        pair_feats = expand_pair(a, b, lookup)
        for c in all_keys:
            if c in (a, b):
                continue
            if base_name(c) in excluded:
                continue
            triple_key = f"{a}+{b}+{c}"
            merged = pair_feats + lookup[c]
            merged = list(dict.fromkeys(merged))
            triple_map[triple_key] = merged
    return triple_map

def triple_sets_for_horizon(horizon: int):
    h_key = f"h_{horizon}"
    if h_key not in selected_pairs:
        raise ValueError(f"Unsupported horizon {horizon}")
    return build_triples_for_pairs(selected_pairs[h_key], raw_features)

# ------------------------------------------------------------------
# 4. Precompute (optional)
# ------------------------------------------------------------------
pair_feature_sets = {h: pair_sets_for_horizon(int(h.split('_')[1])) for h in selected_pairs}
triple_feature_sets = {h: triple_sets_for_horizon(int(h.split('_')[1])) for h in selected_pairs}

# ------------------------------------------------------------------
# 5. Sanity check: missing atomic lists?
# ------------------------------------------------------------------
referenced = {n for pairs in selected_pairs.values() for (a,b) in pairs for n in (a,b)}
missing = [n for n in referenced if n not in raw_features]
if missing:
    print("WARNING: Missing atomic lists:", missing)

# ------------------------------------------------------------------
# 6. Example usage
# ------------------------------------------------------------------
# Get pair feature sets for horizon 5
h5_pairs = pair_sets_for_horizon(5)
# Iterate and train:
# for combo_name, feats in h5_pairs.items():
#     X = df[feats]
#     y = df["Return_5"]
#     ...

# Get triple feature sets for horizon 5
h5_triples = triple_sets_for_horizon(5)
# for combo_name, feats in h5_triples.items():
#     ...

print("Pairs (h_5):", len(h5_pairs), "Triples (h_5):", len(h5_triples))


Pairs (h_5): 5 Triples (h_5): 195


In [91]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.5]
arch_types = ['deep']#['deep', 'moderate', 'shallow']
#n = len(raw_cols_list)
results = []
lb = 8
returns = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
name_to_combo = { " + ".join(c): c for c in valid_combos }
raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()

df = extract(ticker, returns, lb, raw_all, windows=[10, 25])

for ticker in tickers:

    if ticker == 'QQQ':
            returns = [2, 3, 4, 5, 8, 10, 15, 20]#[4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
    else:
        returns = [4, 6, 8, 10, 15]

    for r in returns:
        
        triples = triple_sets_for_horizon(r)
            
        for combo_name, cols in triples.items():
            
            #df = extract(ticker, returns, lb, cat_cols_all, windows=[10, 25])
            df_ph = df.copy()
            df_ph = df_ph.iloc[r:].copy()
            return_col = f"Return_{r}"
            return_perc_col = f"Return%_{r}"
            model_key = f"QQQ_{r}"
            counts = df_ph[return_col].value_counts()
            neg = counts.get(0, 0)
            pos = counts.get(1, 1)  # prevent division by zero
            #print(f'{pos} | {neg}')
            scale_pos_weight = neg / pos


            rets = df_ph[return_perc_col].copy()
            # Split into negative and positive returns
            neg = rets[rets < 0]
            pos = rets[rets > 0]

            # Calculate dynamic thresholds
            neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
            pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude
            #print(f'{pos_cutoff} | {neg_cutoff}')
            filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()
            counts = filtered[return_col].value_counts()
            negf = counts.get(0, 0)
            posf = counts.get(1, 1)  # prevent division by zero
            #print(f'{posf} | {negf}')
            scale_pos_weightf = negf / posf

            dfs = [df_ph, filtered]
            df_names = ['orig', 'filt']
            weights = [scale_pos_weight, scale_pos_weightf]
            dfs = [filtered]
            df_names = ['filt']
            weights = [scale_pos_weightf]

            for arch in arch_types:

                for dataframe, name, weight in zip(dfs, df_names, weights):

                    # Choose evaluation metric
                    opt = 'matthews_corrcoef'

                    # Combine features with return column, drop missing
                    #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                    cols = list(dict.fromkeys(cols))
                    used_cols = cols + [return_col]
                    df_model = dataframe[used_cols].dropna()
                    #print(df_ph['Date'].iloc[0])
                    
                    df_indicators = df_model[cols]
                    df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                    df_predict = df_model[return_col]
                    test_size = 100 
                    val_size = 300
                    perf_size = 100
                    
                    print(f"Results for {name} - {combo_name} | {arch} | {ticker}_{r}")
                    xg_metrics, best_xg_model = optimize_ttv2(df_indicators, df_predict, thresh, opt, weight, arch, test_size, val_size, perf_size, return_metrics=True)
                    metrics = list(xg_metrics.values())[0]
                    pscore = metrics.get('PosFb', 0)
                    nscore = metrics.get('NegFb', 0)
                    row = {'df': name, 'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                    results.append(row)
                    # build one flat record
                    """
                    if pscore >= .8 and nscore >= .8:
                        row = {'df': name, 'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                        results.append(row)
                    """
                    print('---------------------------')

Results for filt - raw_trend_fast+raw_trend_ratio_slow_EMA25+raw_duration_slow | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1453, 'PosTot': 59, 'NegTot': 41, 'PosRec': 0.63, 'PosPrec': 0.74, 'NegRec': 0.68, 'NegPrec': 0.56, 'PosFb': 0.7, 'NegFb': 0.6}
---------------------------
Results for filt - raw_trend_fast+raw_trend_ratio_slow_EMA25+raw_duration_slow_EMA10 | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1453, 'PosTot': 59, 'NegTot': 41, 'PosRec': 0.7, 'PosPrec': 0.72, 'NegRec': 0.61, 'NegPrec': 0.58, 'PosFb': 0.71, 'NegFb': 0.59}
---------------------------
Results for filt - raw_trend_fast+raw_trend_ratio_slow_EMA25+raw_duration_slow_EMA25 | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1453, 'PosTot': 59, 'NegTot': 41, 'PosRec': 0.61, 'PosPrec': 0.78, 'NegRec': 0.75, 'NegPrec': 0.57, 'PosFb': 0.71, 'NegFb': 0.62}
---------------------------
Results for filt - raw_trend_fast+raw_trend_ratio_slow_EMA25+raw_duration_moderate | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1453, 'PosTot': 59, 'NegT

In [23]:
results_df = pd.DataFrame(results)
results_df.to_csv('seasons.csv')

# Seasonality Experimentation

In [36]:
def add_cyclic_seasonality(
    df: pd.DataFrame,
    date_col: str = 'Date',
    add_weekly: bool = True,
    add_month: bool = True,
    add_quarter: bool = True,
    add_year: bool = True,
    add_day_of_year: bool = False,
    mode: str = 'calendar',          # 'trading', 'calendar', or 'both'
    prefix: str = 'cyc_'
) -> pd.DataFrame:
    """
    Add cyclic (sin/cos) seasonality features.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a date column.
    date_col : str
        Name of the datetime column.
    add_weekly, add_month, add_quarter, add_year, add_day_of_year : bool
        Toggles for which seasonal cycles to create.
    mode : {'trading','calendar','both'}
        - 'trading': position = trading-day index / trading-days-in-period (resets at dataset slice start)
        - 'calendar': position = calendar progress within period (day-based; retains true mid-period phase)
        - 'both': create both sets (with suffixes _trade and _cal)
    prefix : str
        Prefix for created feature names.

    Returns
    -------
    pd.DataFrame
        Copy of df with new seasonality features appended (original row order preserved).
    """
    if date_col not in df.columns:
        raise ValueError(f"'{date_col}' not found in DataFrame.")
    if mode not in {'trading', 'calendar', 'both'}:
        raise ValueError("mode must be 'trading', 'calendar', or 'both'.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Preserve original order / index
    original_index = out.index
    work = out.sort_values(date_col).reset_index(drop=False)
    idx_col = work.columns[0]

    # Precompute basic date parts
    work['_year']    = work[date_col].dt.year
    work['_month']   = work[date_col].dt.month
    work['_quarter'] = work[date_col].dt.quarter
    work['_weekday'] = work[date_col].dt.weekday
    work['_doy']     = work[date_col].dt.dayofyear
    work['_is_leap'] = work[date_col].dt.is_leap_year

    # ---------- WEEKLY (trading week: Mon-Fri) ----------
    if add_weekly:
        week_pos = work['_weekday'] / 5.0  # Monday=0.0, Friday≈0.8 (if only Mon-Fri present)
        work[f'{prefix}week_sin'] = np.sin(2 * np.pi * week_pos)
        work[f'{prefix}week_cos'] = np.cos(2 * np.pi * week_pos)

    # Helper to safely divide
    def _safe_div(num, den):
        return np.where(den == 0, 0.0, num / np.where(den == 0, 1, den))

    # ---------- MONTH ----------
    if add_month:
        if mode in {'trading', 'both'}:
            grp_m = [work['_year'], work['_month']]
            idx_m = work.groupby(grp_m).cumcount()
            cnt_m = work.groupby(grp_m)[date_col].transform('count')
            pos_m_trade = _safe_div(idx_m, cnt_m)
            work[f'{prefix}month_trade_sin'] = np.sin(2 * np.pi * pos_m_trade)
            work[f'{prefix}month_trade_cos'] = np.cos(2 * np.pi * pos_m_trade)

        if mode in {'calendar', 'both'}:
            dom = work[date_col].dt.day           # 1..days_in_month
            dim = work[date_col].dt.days_in_month
            pos_m_cal = _safe_div(dom - 1, dim - 1)
            work[f'{prefix}month_sin'] = np.sin(2 * np.pi * pos_m_cal)
            work[f'{prefix}month_cos'] = np.cos(2 * np.pi * pos_m_cal)

    # ---------- QUARTER ----------
    if add_quarter:
        if mode in {'trading', 'both'}:
            grp_q = [work['_year'], work['_quarter']]
            idx_q = work.groupby(grp_q).cumcount()
            cnt_q = work.groupby(grp_q)[date_col].transform('count')
            pos_q_trade = _safe_div(idx_q, cnt_q)
            work[f'{prefix}quarter_trade_sin'] = np.sin(2 * np.pi * pos_q_trade)
            work[f'{prefix}quarter_trade_cos'] = np.cos(2 * np.pi * pos_q_trade)

        if mode in {'calendar', 'both'}:
            q = work['_quarter']
            y = work['_year']
            quarter_start = pd.to_datetime({'year': y, 'month': (q - 1) * 3 + 1, 'day': 1})
            quarter_end = quarter_start + pd.offsets.QuarterEnd(0)
            day_in_q = (work[date_col] - quarter_start).dt.days
            days_q = (quarter_end - quarter_start).dt.days
            pos_q_cal = _safe_div(day_in_q, days_q)
            work[f'{prefix}quarter_sin'] = np.sin(2 * np.pi * pos_q_cal)
            work[f'{prefix}quarter_cos'] = np.cos(2 * np.pi * pos_q_cal)

    # ---------- YEAR ----------
    if add_year:
        if mode in {'trading', 'both'}:
            grp_y = work['_year']
            idx_y = work.groupby(grp_y).cumcount()
            cnt_y = work.groupby(grp_y)[date_col].transform('count')
            pos_y_trade = _safe_div(idx_y, cnt_y)
            work[f'{prefix}year_trade_sin'] = np.sin(2 * np.pi * pos_y_trade)
            work[f'{prefix}year_trade_cos'] = np.cos(2 * np.pi * pos_y_trade)

        if mode in {'calendar', 'both'}:
            doy = work['_doy']
            year_len = np.where(work['_is_leap'], 366, 365)
            pos_y_cal = _safe_div(doy - 1, year_len - 1)
            work[f'{prefix}year_sin'] = np.sin(2 * np.pi * pos_y_cal)
            work[f'{prefix}year_cos'] = np.cos(2 * np.pi * pos_y_cal)

    # ---------- DAY OF YEAR (explicit) ----------
    if add_day_of_year:
        year_len = np.where(work['_is_leap'], 366, 365)
        work[f'{prefix}doy_sin'] = np.sin(2 * np.pi * work['_doy'] / year_len)
        work[f'{prefix}doy_cos'] = np.cos(2 * np.pi * work['_doy'] / year_len)

    # Drop temp columns
    temp_cols = [c for c in work.columns if c.startswith('_')]
    work = work.drop(columns=temp_cols)

    # Identify newly created features
    new_cols = [c for c in work.columns if c not in out.columns and c != idx_col]

    # Map back to original order
    work = work.set_index(idx_col)
    out[new_cols] = work.loc[original_index, new_cols]

    return out

empty = []
weekly = ['Date', 'cyc_week_sin', 'cyc_week_cos']
monthly = ['cyc_month_sin', 'cyc_month_cos']
quarterly = ['cyc_quarter_sin', 'cyc_quarter_cos']
yearly = ['cyc_year_sin', 'cyc_year_cos']
all_seasons = ['cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos',
               'cyc_year_sin', 'cyc_year_cos']

season_lists = [empty, weekly, monthly, quarterly, yearly]
season_names = ['empty', 'weekly', 'monthly', 'quarterly', 'yearly']

season_lists = [all_seasons]
season_names = ['all_seasons']

In [38]:
df[weekly].tail(20)

,Date,cyc_week_sin,cyc_week_cos
268,2018-08-13,0.000000,1.000000
267,2018-08-10,-0.951057,0.309017
266,2018-08-09,-0.587785,-0.809017
265,2018-08-08,0.587785,-0.809017
264,2018-08-07,0.951057,0.309017
263,2018-08-06,0.000000,1.000000
262,2018-08-03,-0.951057,0.309017
261,2018-08-02,-0.587785,-0.809017
260,2018-08-01,0.587785,-0.809017
259,2018-07-31,0.951057,0.309017


In [22]:
tickers = ['QQQ']#, 'NVDA', "AAPL", "MSFT", "TSLA", "AMZN", "AVGO", "META", "GOOGL", "COST", "NFLX"]
thresh = [.5]
arch_types = ['deep']#['deep', 'moderate', 'shallow']
#n = len(raw_cols_list)
#results = []
lb = 8
returns = [2, 3, 4, 5, 8, 10, 15, 20]

raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()
df = extract(ticker, returns, lb, raw_all, windows=[10, 25])
df = add_cyclic_seasonality(df, 
                                 date_col='Date',
                                 add_weekly=True,
                                 add_month=True,
                                 add_quarter=True,
                                 add_year=True,
                                 add_day_of_year=False,
                                 mode='calendar',    # or 'trading' or 'both'
                                 prefix='cyc_')


selected_combos = {
    "h_2": {
    'raw_trend_fast+raw_trend_ratio_slow_25': raw_trend_fast+raw_trend_ratio_slow_25,
    'raw_duration_slow_10+raw_trend_moderate+raw_trend_ratio_slow_10': raw_duration_slow_10+raw_trend_moderate+raw_trend_ratio_slow_10,
    'raw_trend_ratio_moderate_25+raw_volatility_moderate_25': raw_trend_ratio_moderate_25+raw_volatility_moderate_25,
    'raw_trend_ratio_fast_25+raw_momentum_moderate_10+raw_momentum_fast_10': raw_trend_ratio_fast_25+raw_momentum_moderate_10+raw_momentum_fast_10,
    'raw_trend_ratio_moderate_10+raw_momentum_moderate_25': raw_trend_ratio_moderate_10+raw_momentum_moderate_25,
    },
    "h_3": {
    'raw_trend_ratio_moderate_25+raw_volatility_moderate_25+raw_trend_fast': raw_trend_ratio_moderate_25+raw_volatility_moderate_25+raw_trend_fast,
    'raw_trend_fast+raw_trend_ratio_slow_25+raw_trend_ratio_moderate': raw_trend_fast+raw_trend_ratio_slow_25+raw_trend_ratio_moderate,
    'raw_duration_fast+raw_trend_slow': raw_duration_fast+raw_trend_slow,
    'raw_trend_ratio_slow+raw_momentum_slow+raw_momentum_moderate_25': raw_trend_ratio_slow+raw_momentum_slow+raw_momentum_moderate_25,
    'raw_trend_ratio_fast_25+raw_momentum_moderate_25+raw_trend_slow': raw_trend_ratio_fast_25+raw_momentum_moderate_25+raw_trend_slow,
    },
    "h_4": {
    'raw_trend_slow+raw_trend_ratio_fast_25+raw_trend_ratio_moderate_25': raw_trend_slow+raw_trend_ratio_fast_25+raw_trend_ratio_moderate_25,
    'raw_trend_slow+raw_trend_ratio_moderate_10+raw_duration_slow': raw_trend_slow+raw_trend_ratio_moderate_10+raw_duration_slow,
    'raw_trend_fast+raw_trend_ratio_moderate_25+raw_trend_moderate_10': raw_trend_fast+raw_trend_ratio_moderate_25+raw_trend_moderate_10,
    'raw_duration_moderate_10+raw_volatility_moderate_25': raw_duration_moderate_10+raw_volatility_moderate_25,
    'raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_slow_10': raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_slow_10,
    },
    "h_5": {
    'raw_duration_slow_10+raw_trend_ratio_fast_25': raw_duration_slow_10+raw_trend_ratio_fast_25,
    'raw_duration_slow+raw_trend_ratio_fast_25': raw_duration_slow+raw_trend_ratio_fast_25,
    'raw_trend_slow+raw_trend_ratio_slow+raw_trend_ratio_fast_10': raw_trend_slow+raw_trend_ratio_slow+raw_trend_ratio_fast_10,
    'raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_fast': raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_fast,
    'raw_trend_moderate_25+raw_trend_ratio_moderate_25': raw_trend_moderate_25+raw_trend_ratio_moderate_25,
    },
    "h_8": {
    'raw_trend_fast+raw_trend_ratio_moderate+raw_trend_slow_25': raw_trend_fast+raw_trend_ratio_moderate+raw_trend_slow_25,
    'raw_trend_fast_10+raw_trend_ratio_slow_25': raw_trend_fast_10+raw_trend_ratio_slow_25,
    'raw_trend_ratio_slow_10+raw_momentum_moderate': raw_trend_ratio_slow_10+raw_momentum_moderate,
    'raw_volatility_moderate_25+raw_momentum_slow_25+raw_trend_ratio_slow_25': raw_volatility_moderate_25+raw_momentum_slow_25+raw_trend_ratio_slow_25,
    'raw_trend_slow+raw_trend_slow_25': raw_trend_slow+raw_trend_slow_25,
    },
    "h_10": {
    'raw_trend_ratio_slow_25+raw_volatility_moderate_25': raw_trend_ratio_slow_25+raw_volatility_moderate_25,
    'raw_duration_moderate+raw_trend_slow': raw_duration_moderate+raw_trend_slow,
    'raw_duration_moderate_25+raw_trend_slow_10': raw_duration_moderate_25+raw_trend_slow_10,
    'raw_trend_ratio_moderate+raw_trend_ratio_fast+raw_momentum_slow_10': raw_trend_ratio_moderate+raw_trend_ratio_fast+raw_momentum_slow_10,
    'raw_trend_ratio_slow_10+raw_momentum_moderate_10': raw_trend_ratio_slow_10+raw_momentum_moderate_10,
    },
    "h_15": {
    'raw_trend_slow_10+raw_trend_ratio_moderate_25+raw_volatility_fast': raw_trend_slow_10+raw_trend_ratio_moderate_25+raw_volatility_fast,
    'raw_duration_moderate+raw_trend_slow_25+raw_momentum_slow_25': raw_duration_moderate+raw_trend_slow_25+raw_momentum_slow_25,
    'raw_trend_ratio_moderate_10+raw_momentum_moderate_25+raw_trend_slow_10': raw_trend_ratio_moderate_10+raw_momentum_moderate_25+raw_trend_slow_10,
    'raw_trend_ratio_fast+raw_momentum_moderate+raw_duration_slow_25': raw_trend_ratio_fast+raw_momentum_moderate+raw_duration_slow_25,
    'raw_trend_ratio_fast_10+raw_volatility_moderate+raw_trend_ratio_moderate': raw_trend_ratio_fast_10+raw_volatility_moderate+raw_trend_ratio_moderate,
    },
    "h_20": {
    'raw_duration_slow+raw_momentum_moderate_25+raw_trend_moderate_10': raw_duration_slow+raw_momentum_moderate_25+raw_trend_moderate_10,
    'raw_trend_ratio_moderate_25+raw_momentum_moderate_25': raw_trend_ratio_moderate_25+raw_momentum_moderate_25,
    'raw_trend_slow_25+raw_momentum_moderate_10': raw_trend_slow_25+raw_momentum_moderate_10,
    'raw_trend_ratio_moderate+raw_volatility_moderate+raw_duration_moderate_25': raw_trend_ratio_moderate+raw_volatility_moderate+raw_duration_moderate_25,
    'raw_trend_moderate+raw_trend_ratio_fast_25+raw_momentum_moderate_25': raw_trend_moderate+raw_trend_ratio_fast_25+raw_momentum_moderate_25,
    },
}

def get_model_set(horizon):
    if horizon in [2]:
        key = "h_2"
    elif horizon in [3]:
        key = "h_3"
    elif horizon in [4]:
        key = "h_4"
    elif horizon in [5]:
        key = "h_5" 
    elif horizon in [8]:
        key = "h_8"
    elif horizon in [10]:
        key = "h_10"
    elif horizon in [15]:
        key = "h_15"
    elif horizon in [20]:
        key = "h_20"
    else:
        raise ValueError(f"Unsupported horizon: {horizon}")
    
    return selected_combos[key]

for ticker in tickers:

    if ticker == 'QQQ':
            returns = [2, 3, 4, 5, 8, 10, 15, 20]#[4, 5, 6, 8, 10, 15, 20, 25, 30, 45, 60, 75, 90]
    else:
        returns = [4, 6, 8, 10, 15]

    for r in returns:
        
        selected_models = get_model_set(r)
        
        # all index-combos of length r
        for name, cols in selected_models.items():

            for season_list, season_name in zip(season_lists, season_names):

                cols_bl = cols.copy()
            
                #df = extract(ticker, returns, lb, cat_cols_all, windows=[10, 25])
                df_ph = df.copy()
                df_ph = df_ph.iloc[r:].copy()
                return_col = f"Return_{r}"
                return_perc_col = f"Return%_{r}"
                model_key = f"QQQ_{r}"
                counts = df_ph[return_col].value_counts()
                neg = counts.get(0, 0)
                pos = counts.get(1, 1)  # prevent division by zero
                #print(f'{pos} | {neg}')
                scale_pos_weight = neg / pos

                rets = df_ph[return_perc_col].copy()
                # Split into negative and positive returns
                neg = rets[rets < 0]
                pos = rets[rets > 0]

                # Calculate dynamic thresholds
                neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
                pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude
                #print(f'{pos_cutoff} | {neg_cutoff}')
                filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()
                counts = filtered[return_col].value_counts()
                negf = counts.get(0, 0)
                posf = counts.get(1, 1)  # prevent division by zero
                #print(f'{posf} | {negf}')
                scale_pos_weightf = negf / posf

                dfs = [df_ph, filtered]
                df_names = ['orig', 'filt']
                weights = [scale_pos_weight, scale_pos_weightf]
                dfs = [filtered]
                df_names = ['filt']
                weights = [scale_pos_weightf]

                for arch in arch_types:

                    for dataframe, na, weight in zip(dfs, df_names, weights):

                        # Choose evaluation metric
                        opt = 'matthews_corrcoef'

                        # Combine features with return column, drop missing
                        #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                        cols_bl += season_list
                        cols_bl = list(dict.fromkeys(cols_bl))
                        print(cols_bl)
                        used_cols = cols_bl + [return_col]
                        df_model = dataframe[used_cols].dropna()
                        #print(df_ph['Date'].iloc[0])
                        
                        df_indicators = df_model[cols_bl]
                        df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                        df_predict = df_model[return_col]
                        test_size = 100 
                        val_size = 300
                        perf_size = 100
                        
                        print(f"Results for {season_name} - {name} | {arch} | {ticker}_{r}")
                        xg_metrics, best_xg_model = optimize_ttv2(df_indicators, df_predict, thresh, opt, weight, arch, test_size, val_size, perf_size, return_metrics=True)
                        metrics = list(xg_metrics.values())[0]
                        pscore = metrics.get('PosFb', 0)
                        nscore = metrics.get('NegFb', 0)
                        row = {'df': season_name, 'name': name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                        results.append(row)
                        # build one flat record
                        """
                        if pscore >= .8 and nscore >= .8:
                            row = {'df': name, 'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                            results.append(row)
                        """
                        print('---------------------------')

['minus_DI', 'plus_DI', '100_SMA_200_EMA25', '100_EMA_200_EMA25', '50_SMA_100_EMA25', '50_EMA_100_EMA25', '25_EMA_50_EMA25', '50_SMA_200_EMA25', '50_EMA_200_EMA25', 'cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos', 'cyc_year_sin', 'cyc_year_cos']
Results for all_seasons - raw_trend_fast+raw_trend_ratio_slow_25 | deep | QQQ_2
  Threshold 0.5: {'TT_Len': 1452, 'PosTot': 59, 'NegTot': 41, 'PosRec': 0.81, 'PosPrec': 0.64, 'NegRec': 0.34, 'NegPrec': 0.56, 'PosFb': 0.69, 'NegFb': 0.46}
---------------------------
['Min_60_Rows_Since_EMA10', 'Min_120_Rows_Since_EMA10', 'num_days_100_EMA10', 'num_days_200_EMA10', 'ADX', 'QQQ_SMA_25', 'QQQ_SMA_50', '100_SMA_200_EMA10', '100_EMA_200_EMA10', '50_SMA_100_EMA10', '50_EMA_100_EMA10', '25_EMA_50_EMA10', '50_SMA_200_EMA10', '50_EMA_200_EMA10', 'cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos', 'cyc_year_sin', 'cyc_year_cos']
Results for all_seaso

# Treasuries and QQEW

In [28]:
# QQEW
tickerSymbol = 'QQEW' # comparative returns over 3, 5, 10, 20 days

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)
start_date = (datetime.today() - relativedelta(years=lb)).strftime('%Y-%m-%d')

tickerDf = tickerData.history(period='1d', start=start_date)

# Resetting the index will turn the Date index into a column
df_qqew = tickerDf.reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]

# 5yr Treasury
tickerSymbol = '^FVX' # raw value and slope over 5, 20, 50, 

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)
start_date = (datetime.today() - relativedelta(years=lb)).strftime('%Y-%m-%d')

tickerDf = tickerData.history(period='1d', start=start_date)

# Resetting the index will turn the Date index into a column
df_5yr = tickerDf.reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]


# 10yr Treasury
tickerSymbol = '^TNX'

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)
start_date = (datetime.today() - relativedelta(years=lb)).strftime('%Y-%m-%d')

tickerDf = tickerData.history(period='1d', start=start_date)

# Resetting the index will turn the Date index into a column
df_10yr = tickerDf.reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]


# 30yr Treasury
tickerSymbol = '^TYX'

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)
start_date = (datetime.today() - relativedelta(years=lb)).strftime('%Y-%m-%d')

tickerDf = tickerData.history(period='1d', start=start_date)

# Resetting the index will turn the Date index into a column
df_30yr = tickerDf.reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]


In [ ]:
results = []

selected_combos = {
    "h_2": {
    'raw_trend_fast+raw_trend_ratio_slow_25': raw_trend_fast+raw_trend_ratio_slow_25,
    'raw_duration_slow_10+raw_trend_moderate+raw_trend_ratio_slow_10': raw_duration_slow_10+raw_trend_moderate+raw_trend_ratio_slow_10,
    'raw_trend_ratio_moderate_25+raw_volatility_moderate_25': raw_trend_ratio_moderate_25+raw_volatility_moderate_25,
    'raw_trend_ratio_fast_25+raw_momentum_moderate_10+raw_momentum_fast_10': raw_trend_ratio_fast_25+raw_momentum_moderate_10+raw_momentum_fast_10,
    'raw_trend_ratio_moderate_10+raw_momentum_moderate_25': raw_trend_ratio_moderate_10+raw_momentum_moderate_25,
    },
    "h_3": {
    'raw_trend_ratio_moderate_25+raw_volatility_moderate_25+raw_trend_fast': raw_trend_ratio_moderate_25+raw_volatility_moderate_25+raw_trend_fast,
    'raw_trend_fast+raw_trend_ratio_slow_25+raw_trend_ratio_moderate': raw_trend_fast+raw_trend_ratio_slow_25+raw_trend_ratio_moderate,
    'raw_duration_fast+raw_trend_slow': raw_duration_fast+raw_trend_slow+all_seasons,
    'raw_trend_ratio_slow+raw_momentum_slow+raw_momentum_moderate_25': raw_trend_ratio_slow+raw_momentum_slow+raw_momentum_moderate_25+monthly,
    'raw_trend_ratio_fast_25+raw_momentum_moderate_25+raw_trend_slow': raw_trend_ratio_fast_25+raw_momentum_moderate_25+raw_trend_slow,
    },
    "h_4": {
    'raw_trend_slow+raw_trend_ratio_fast_25+raw_trend_ratio_moderate_25': raw_trend_slow+raw_trend_ratio_fast_25+raw_trend_ratio_moderate_25,
    'raw_trend_slow+raw_trend_ratio_moderate_10+raw_duration_slow': raw_trend_slow+raw_trend_ratio_moderate_10+raw_duration_slow,
    'raw_trend_fast+raw_trend_ratio_moderate_25+raw_trend_moderate_10': raw_trend_fast+raw_trend_ratio_moderate_25+raw_trend_moderate_10,
    'raw_duration_moderate_10+raw_volatility_moderate_25': raw_duration_moderate_10+raw_volatility_moderate_25+weekly,
    'raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_slow_10': raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_slow_10,
    },
    "h_5": {
    'raw_duration_slow_10+raw_trend_ratio_fast_25': raw_duration_slow_10+raw_trend_ratio_fast_25+weekly,
    'raw_duration_slow+raw_trend_ratio_fast_25': raw_duration_slow+raw_trend_ratio_fast_25,
    'raw_trend_slow+raw_trend_ratio_slow+raw_trend_ratio_fast_10': raw_trend_slow+raw_trend_ratio_slow+raw_trend_ratio_fast_10+weekly,
    'raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_fast': raw_trend_slow_25+raw_momentum_moderate+raw_trend_ratio_fast,
    'raw_trend_moderate_25+raw_trend_ratio_moderate_25': raw_trend_moderate_25+raw_trend_ratio_moderate_25+monthly,
    },
    "h_8": {
    'raw_trend_fast+raw_trend_ratio_moderate+raw_trend_slow_25': raw_trend_fast+raw_trend_ratio_moderate+raw_trend_slow_25,
    'raw_trend_fast_10+raw_trend_ratio_slow_25': raw_trend_fast_10+raw_trend_ratio_slow_25,
    'raw_trend_ratio_slow_10+raw_momentum_moderate': raw_trend_ratio_slow_10+raw_momentum_moderate+weekly,
    'raw_volatility_moderate_25+raw_momentum_slow_25+raw_trend_ratio_slow_25': raw_volatility_moderate_25+raw_momentum_slow_25+raw_trend_ratio_slow_25+weekly,
    'raw_trend_slow+raw_trend_slow_25': raw_trend_slow+raw_trend_slow_25,
    },
    "h_10": {
    'raw_trend_ratio_slow_25+raw_volatility_moderate_25': raw_trend_ratio_slow_25+raw_volatility_moderate_25+yearly,
    'raw_duration_moderate+raw_trend_slow': raw_duration_moderate+raw_trend_slow,
    'raw_duration_moderate_25+raw_trend_slow_10': raw_duration_moderate_25+raw_trend_slow_10+weekly,
    'raw_trend_ratio_moderate+raw_trend_ratio_fast+raw_momentum_slow_10': raw_trend_ratio_moderate+raw_trend_ratio_fast+raw_momentum_slow_10+weekly,
    'raw_trend_ratio_slow_10+raw_momentum_moderate_10': raw_trend_ratio_slow_10+raw_momentum_moderate_10+monthly,
    },
    "h_15": {
    'raw_trend_slow_10+raw_trend_ratio_moderate_25+raw_volatility_fast': raw_trend_slow_10+raw_trend_ratio_moderate_25+raw_volatility_fast+weekly,
    'raw_duration_moderate+raw_trend_slow_25+raw_momentum_slow_25': raw_duration_moderate+raw_trend_slow_25+raw_momentum_slow_25,
    'raw_trend_ratio_moderate_10+raw_momentum_moderate_25+raw_trend_slow_10': raw_trend_ratio_moderate_10+raw_momentum_moderate_25+raw_trend_slow_10,
    'raw_trend_ratio_fast+raw_momentum_moderate+raw_duration_slow_25': raw_trend_ratio_fast+raw_momentum_moderate+raw_duration_slow_25,
    'raw_trend_ratio_fast_10+raw_volatility_moderate+raw_trend_ratio_moderate': raw_trend_ratio_fast_10+raw_volatility_moderate+raw_trend_ratio_moderate,
    },
    "h_20": {
    'raw_duration_slow+raw_momentum_moderate_25+raw_trend_moderate_10': raw_duration_slow+raw_momentum_moderate_25+raw_trend_moderate_10+quarterly,
    'raw_trend_ratio_moderate_25+raw_momentum_moderate_25': raw_trend_ratio_moderate_25+raw_momentum_moderate_25,
    'raw_trend_slow_25+raw_momentum_moderate_10': raw_trend_slow_25+raw_momentum_moderate_10,
    'raw_trend_ratio_moderate+raw_volatility_moderate+raw_duration_moderate_25': raw_trend_ratio_moderate+raw_volatility_moderate+raw_duration_moderate_25+weekly,
    'raw_trend_moderate+raw_trend_ratio_fast_25+raw_momentum_moderate_25': raw_trend_moderate+raw_trend_ratio_fast_25+raw_momentum_moderate_25+weekly,
    },
}

def get_model_set(horizon):
    if horizon in [2]:
        key = "h_2"
    elif horizon in [3]:
        key = "h_3"
    elif horizon in [4]:
        key = "h_4"
    elif horizon in [5]:
        key = "h_5" 
    elif horizon in [8]:
        key = "h_8"
    elif horizon in [10]:
        key = "h_10"
    elif horizon in [15]:
        key = "h_15"
    elif horizon in [20]:
        key = "h_20"
    else:
        raise ValueError(f"Unsupported horizon: {horizon}")
    
    return selected_combos[key]

for ticker in tickers:
    
    df = extract(ticker, returns, lb, raw_all, windows=[10,25])
    df = add_cyclic_seasonality(df, 
                                 date_col='Date',
                                 add_weekly=True,
                                 add_month=True,
                                 add_quarter=True,
                                 add_year=True,
                                 add_day_of_year=False,
                                 mode='calendar',    # or 'trading' or 'both'
                                 prefix='cyc_')

    for r in returns:

        selected_models = get_model_set(r)
        
        # all index-combos of length r
        for name, cols in selected_models.items():

            for season_list, season_name in zip(season_lists, season_names):

                cols_bl = cols.copy()
            
                #df = extract(ticker, returns, lb, cat_cols_all, windows=[10, 25])
                df_ph = df.copy()
                df_ph = df_ph.iloc[r:].copy()
                return_col = f"Return_{r}"
                return_perc_col = f"Return%_{r}"
                model_key = f"QQQ_{r}"
                counts = df_ph[return_col].value_counts()
                neg = counts.get(0, 0)
                pos = counts.get(1, 1)  # prevent division by zero
                #print(f'{pos} | {neg}')
                scale_pos_weight = neg / pos

                rets = df_ph[return_perc_col].copy()
                # Split into negative and positive returns
                neg = rets[rets < 0]
                pos = rets[rets > 0]

                # Calculate dynamic thresholds
                neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
                pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude
                #print(f'{pos_cutoff} | {neg_cutoff}')
                filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()
                counts = filtered[return_col].value_counts()
                negf = counts.get(0, 0)
                posf = counts.get(1, 1)  # prevent division by zero
                #print(f'{posf} | {negf}')
                scale_pos_weightf = negf / posf

                dfs = [df_ph, filtered]
                df_names = ['orig', 'filt']
                weights = [scale_pos_weight, scale_pos_weightf]
                dfs = [filtered]
                df_names = ['filt']
                weights = [scale_pos_weightf]

                for arch in arch_types:

                    for dataframe, na, weight in zip(dfs, df_names, weights):

                        # Choose evaluation metric
                        opt = 'matthews_corrcoef'

                        # Combine features with return column, drop missing
                        #cols += ['Close_slope10', 'Close_slope25', 'Close_slope50']
                        cols_bl += season_list
                        cols_bl = list(dict.fromkeys(cols_bl))
                        print(cols_bl)
                        used_cols = cols_bl + [return_col]
                        df_model = dataframe[used_cols].dropna()
                        #print(df_ph['Date'].iloc[0])
                        
                        df_indicators = df_model[cols_bl]
                        df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
                        df_predict = df_model[return_col]
                        test_size = 100 
                        val_size = 300
                        perf_size = 100
                        
                        print(f"Results for {season_name} - {name} | {arch} | {ticker}_{r}")
                        xg_metrics, best_xg_model = optimize_ttv2(df_indicators, df_predict, thresh, opt, weight, arch, test_size, val_size, perf_size, return_metrics=True)
                        metrics = list(xg_metrics.values())[0]
                        pscore = metrics.get('PosFb', 0)
                        nscore = metrics.get('NegFb', 0)
                        row = {'df': season_name, 'name': name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                        results.append(row)
                        # build one flat record
                        """
                        if pscore >= .8 and nscore >= .8:
                            row = {'df': name, 'name': combo_name, 'arch': arch, 'ticker': ticker, 'horizon': r, 'val_size': val_size, **metrics}
                            results.append(row)
                        """
                        print('---------------------------')